In [ ]:
import mlflow.sklearn
import pandas as pd
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset
from notebook_utils import mssparkutils # For Pipeline control

# 1. Load current data from silver layer
df = spark.read.table("lh_CustomerChurn.silver.customerchurn").toPandas()

# 1b. Convert decimal-columns (coming as object/decimal from Spark) into float
decimal_cols = [c for c in df.columns if df[c].apply(lambda x: isinstance(x, Decimal)).any()]
print(f"Decimal-columns converted: {decimal_cols}")
df[decimal_cols] = df[decimal_cols].astype(float)

# 2. Handling Nulls
rows_before = len(df)
df = df.dropna()
print(f"Deleted rows (Nulls): {rows_before - len(df)} von {rows_before}")

# Use current features to make predictions with ML Model XGBoost
X_current = df.drop(columns=['cr8b0_churnstatus', 'cr8b0_customerid', 'cr8b0_customersubscriptionid'])

# 2. Das aktuell produktive Modell aus MLflow laden
# Ersetzt das feste Registrieren durch das Laden des "Champion"-Modells
mlflow.set_experiment("ML_CustomerChurn-1821")
model_uri = "models:/ML_CustomerChurn-1821-xgb/latest" # Oder über Tags gesteuert
pipeline_xgb = mlflow.sklearn.load_model(model_uri)

# 3. Vorhersagen für die heutige Tabelle treffen (Batch Scoring)
df_current['predicted_churn'] = pipeline_xgb.predict(X_current)
df_current['churn_probability'] = pipeline_xgb.predict_proba(X_current)[:, 1]

# Schreibe df_current mit den Vorhersagen zurück in den Gold Layer / Dataverse!

# save relevant data for checking Model drifting
df_predictions = df_current[['cr8b0_customerid', 'predicted_churn', 'churn_probability']].copy()
# Zeitstempel hinzufügen, damit du im Report siehst, von wann die Vorhersage ist
df_predictions['prediction_date'] = pd.Timestamp.now()

# Write Predictins in gold table
spark.createDataFrame(df_predictions).write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("lh_CustomerChurn.gold.customer_churn_predictions")


# 4. Drift-Check with Evidently
try:
    # Versuche die historische Trainingsbasis zu laden (wird im Trainings-Notebook erzeugt)
    df_reference = spark.read.table("lh_CustomerChurn.silver.original_training_data").toPandas()
    X_reference = df_reference.drop(columns=['cr8b0_churnstatus', 'cr8b0_customerid', 'cr8b0_customersubscriptionid'])
    
    # Vorhergesagte Labels für den statistischen Vergleich hinzufügen
    reference_data = X_reference.copy()
    reference_data['prediction'] = pipeline_xgb.predict(X_reference)

    current_data = X_current.copy()
    current_data['prediction'] = df_current['predicted_churn'].values

    # Evidently Report konfigurieren und ausführen
    report = Report(metrics=[DataDriftPreset()])
    report.run(reference_data=reference_data, current_data=current_data)
    
    # HTML Report im Lakehouse-Dateisystem speichern
    report_path = "/lakehouse/default/Files/silver/daily_drift_report.html"
    report.save_html(report_path)
    
    # In MLflow loggen, um die Historie der Reports zu behalten
    with mlflow.start_run(run_name="daily_drift_report"):
        mlflow.log_artifact(report_path)
    
    # Drift-Ergebnis für Pipeline-Steuerung auslesen
    report_dict = report.as_dict()
    drift_detected = report_dict['metrics']['result']['dataset_drift']

except Exception as e:
    # Falls die Tabelle beim allerersten Durchlauf noch nicht existiert
    print(f"⚠️ Hinweis: Drift-Check übersprungen ({str(e)}). Tabelle 'original_training_data' fehlt vermutlich noch.")
    drift_detected = True  # Erzwingt das initiale Training, um die Tabelle anzulegen!


# --- 5. PIPELINE-STEUERUNG (EXIT CODES) ---
if drift_detected:
    print("🚨 DATA DRIFT ERKANNT! Das Modell sollte neu trainiert werden.")
    displayHTML(open(report_path).read()) if 'report_path' in locals() else None
    mssparkutils.notebook.exit("Drift_Detected")
else:
    print("✅ Datenstrukturen sind stabil. Kein Data Drift vorhanden.")
    mssparkutils.notebook.exit("No_Drift")

